#**Employee Churn Analysis**

> This model is designed to pinpoint employees who may be contemplating departure. This enables our Human Resources team to engage proactively and address individual employee concerns with precision.




## **Data Ingestion**

In [ ]:
# Authenticate user and Connect to Bigquery
from google.cloud import bigquery
from google.colab import auth

auth.authenticate_user()

project_id = 'employee-churn-analysis-500216'
client = bigquery.Client(project=project_id)

In [ ]:
# Get the employee dataset and tables
import pandas as pd

dataset_ref = client.dataset('employeedata', project=project_id)

table_hr = client.get_table(dataset_ref.table('hr_data'))
df = client.list_rows(table=table_hr).to_dataframe()

table_new = client.get_table(dataset_ref.table('new_employees'))
df2 = client.list_rows(table=table_new).to_dataframe()

# Data quick check
print(f"Historical cohort dataset shape: {df.shape}")
print(f"New hire prediction dataset shape: {df2.shape}")

Historical cohort dataset shape: (15004, 11)
New hire prediction dataset shape: (100, 11)


## **Distribution Analysis and Multicollinearity Check**

In [ ]:
# Evaluate minority class imbalance
print(df['Quit_the_Company'].value_counts(normalize=True))

Quit_the_Company
0    0.761664
1    0.238336
Name: proportion, dtype: Float64


Employee churn is naturally imbalanced
=> Priotitize optimization metric roc_auc or f1_score over accuracy to avoid false predictions.

In [ ]:
import numpy as np

X_preliminary = df.drop(columns=['Quit_the_Company', 'employee_id'], errors='ignore')
num_cols = X_preliminary.select_dtypes(exclude=['object', 'category']).columns

# Screen for highly collinear numerical features (> 0.85 correlation coefficient)
correlation_matrix = df[num_cols].corr()
high_corr = np.where(np.abs(correlation_matrix) > 0.85)
high_corr_list = [(correlation_matrix.index[x], correlation_matrix.columns[y])
                  for x, y in zip(*high_corr) if x != y and x < y]
print(f"Collinear feature pairs threatening permutation accuracy: {high_corr_list}")

Collinear feature pairs threatening permutation accuracy: []


=> Independent variables are not highly correlated with each other.

In [ ]:
# Set up final working datasets
X = X_preliminary.copy()
y = df['Quit_the_Company']

##**Data Preprocessing**

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Separate categorical and numerical columns
cat_features = X.select_dtypes(include=['object', 'category']).columns
num_features = X.select_dtypes(exclude=['object', 'category']).columns

# Build structured preprocessing layers
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), num_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), cat_features)
])

## **Model Selection**

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Use class_weight='balanced' parameter to prevent minority group classification bias
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Decision Tree': DecisionTreeClassifier(random_state=123, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=123, class_weight='balanced'),
    'Extra Trees': ExtraTreesClassifier(random_state=123, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(random_state=123)
}

results = []
for name, model in models.items():
    pipeline = Pipeline([('pre', preprocessor), ('model', model)])
    # Optimization scoring parameter shifted from accuracy to roc_auc
    score = cross_val_score(pipeline, X, y, cv=5, scoring='roc_auc').mean()
    results.append([name, score])

results_df = pd.DataFrame(results, columns=['Model', 'ROC-AUC Score']).sort_values('ROC-AUC Score', ascending=False)
print(results_df)

                 Model  ROC-AUC Score
2        Random Forest       0.993398
3          Extra Trees       0.992136
4    Gradient Boosting       0.987142
1        Decision Tree       0.974533
0  Logistic Regression       0.804229


=> The model with highest ROC-AUC score is Random Forest.

## **Model training**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.inspection import permutation_importance

# Split data to run strict model out-of-sample error diagnostics
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123, stratify=y)

best_model = RandomForestClassifier(random_state=123, class_weight='balanced')
diagnostic_pipeline = Pipeline([('pre', preprocessor), ('model', best_model)])
diagnostic_pipeline.fit(X_train, y_train)

test_preds = diagnostic_pipeline.predict(X_test)
test_probs = diagnostic_pipeline.predict_proba(X_test)[:, 1]

print("Classification Metrics on Unseen Test Partition:")
print(classification_report(y_test, test_preds))
print(f"Unseen Test ROC-AUC Score: {roc_auc_score(y_test, test_probs)}")

# Run permutation evaluation on test subset to eliminate training-data memory bias
r = permutation_importance(diagnostic_pipeline, X_test, y_test, n_repeats=5, random_state=123)
feature_table = pd.DataFrame({'feature': X.columns, 'importance': r.importances_mean}).sort_values('importance', ascending=False)

Classification Metrics on Unseen Test Partition:
              precision    recall  f1-score   support

         0.0       0.99      1.00      0.99      2286
         1.0       0.99      0.97      0.98       715

    accuracy                           0.99      3001
   macro avg       0.99      0.98      0.99      3001
weighted avg       0.99      0.99      0.99      3001

Unseen Test ROC-AUC Score: 0.9946594962342994


In [ ]:
# Refit on 100% of available historical data
final_pipeline = Pipeline([('pre', preprocessor), ('model', RandomForestClassifier(random_state=123, class_weight='balanced'))])
final_pipeline.fit(X, y)

df['Prediction'] = final_pipeline.predict(X)

# Enforce explicit matrix column alignment on new features before forward scoring
X_new = df2[X.columns]
df2['Prediction'] = final_pipeline.predict(X_new)

In [ ]:
df2.to_gbq('employeedata.pilot_prediction', project_id, if_exists='replace')

/tmp/ipykernel_918/1319070456.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_gbq except for the argument 'destination_table' will be keyword-only.
  df2.to_gbq('employeedata.pilot_prediction', project_id, if_exists='replace')
/tmp/ipykernel_918/1319070456.py:1: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  df2.to_gbq('employeedata.pilot_prediction', project_id, if_exists='replace')
100%|██████████| 1/1 [00:00<00:00, 1714.76it/s]


In [ ]:
feature_table

,feature,importance
0,satisfaction_level,0.185671
2,number_project,0.118427
1,last_evaluation,0.117561
3,average_montly_hours,0.117561
4,time_spend_company,0.097301
8,salary,0.009530
7,Departments,0.009397
5,Work_accident,0.002466
6,promotion_last_5years,0.000200


In [ ]:
feature_table.to_gbq('employeedata.feature_table', project_id, if_exists='replace')

/tmp/ipykernel_918/1152125841.py:1: FutureWarning: Starting with pandas version 3.0 all arguments of to_gbq except for the argument 'destination_table' will be keyword-only.
  feature_table.to_gbq('employeedata.feature_table', project_id, if_exists='replace')
/tmp/ipykernel_918/1152125841.py:1: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  feature_table.to_gbq('employeedata.feature_table', project_id, if_exists='replace')
100%|██████████| 1/1 [00:00<00:00, 8756.38it/s]
